# SELECT Basics

## Overview
`SELECT` is the foundation of every SQL query. It retrieves rows and columns from tables, optionally transforming, renaming, and filtering data inline.

**Full SELECT syntax (SQLite and PostgreSQL):**
```sql
SELECT  [DISTINCT] column1, column2, expression AS alias
FROM    table_name
WHERE   condition
GROUP BY column
HAVING  group_condition
ORDER BY column [ASC|DESC]
LIMIT   n OFFSET k;
```

**Logical execution order** (not the written order — this matters for understanding what you can reference where):

| Step | Clause | What it does |
|---|---|---|
| 1 | `FROM` | Identifies the source table(s) |
| 2 | `WHERE` | Filters individual rows |
| 3 | `GROUP BY` | Groups rows for aggregation |
| 4 | `HAVING` | Filters groups |
| 5 | `SELECT` | Evaluates column expressions and aliases |
| 6 | `ORDER BY` | Sorts the result |
| 7 | `LIMIT` | Caps the number of rows returned |

Because `WHERE` runs before `SELECT`, you **cannot use a SELECT alias in a WHERE clause**. Use a subquery or CTE if you need to filter on a derived value.

**Setup note:** All `01_sql_fundamentals` notebooks use SQLite via Python's built-in `sqlite3` module — no database server required. PostgreSQL syntax differences are noted inline where they matter.

---

In [ ]:
import sqlite3
import pandas as pd

# All notebooks in this folder share the same healthcare schema.
# We recreate it fresh in each notebook so they run independently.

conn = sqlite3.connect(':memory:')
cur  = conn.cursor()

cur.executescript("""
-- Patients table
CREATE TABLE patients (
    patient_id   INTEGER PRIMARY KEY,
    first_name   TEXT,
    last_name    TEXT,
    dob          TEXT,        -- stored as YYYY-MM-DD text (SQLite has no DATE type)
    sex          TEXT CHECK(sex IN ('M','F','O')),
    province     TEXT,
    active       INTEGER DEFAULT 1   -- 1 = active, 0 = inactive
);

INSERT INTO patients VALUES
  (1,  'Aroha',   'Ngata',      '1985-03-12', 'F', 'NB', 1),
  (2,  'Liam',    'Chen',       '1972-11-04', 'M', 'NS', 1),
  (3,  'Fatima',  'Al-Rashid',  '1990-07-22', 'F', 'ON', 1),
  (4,  'James',   'MacLeod',    '1955-01-30', 'M', 'NB', 0),
  (5,  'Sofia',   'Petrov',     '2001-09-15', 'F', 'BC', 1),
  (6,  'Noah',    'Williams',   '1968-06-08', 'M', 'AB', 1),
  (7,  'Mei',     'Zhang',      '1995-04-17', 'F', 'ON', 1),
  (8,  'Ethan',   'Tremblay',   '1980-12-01', 'M', 'QC', 0),
  (9,  'Priya',   'Nair',       '1977-08-25', 'F', 'BC', 1),
  (10, 'Marcus',  'Okafor',     '1993-05-19', 'M', 'ON', 1);
""")
conn.commit()

n = cur.execute('SELECT COUNT(*) FROM patients').fetchone()[0]
print(f'patients table ready — {n} rows')
print()
print(pd.read_sql('SELECT * FROM patients', conn).to_string(index=False))


---
## Basic SELECT: choosing columns

The simplest SELECT retrieves named columns. You can also compute new values inline using expressions, string operators, and `CASE`.


In [ ]:
# --- Select all columns with *
print('=== SELECT * (all columns) ===')
print(pd.read_sql('SELECT * FROM patients', conn).to_string(index=False))

# --- Select specific columns only
print()
print('=== Specific columns only ===')
q = 'SELECT patient_id, first_name, last_name, province FROM patients'
print(pd.read_sql(q, conn).to_string(index=False))


---
## Column expressions and aliases

Use `AS` to rename a column or label a computed expression in the output. Aliases can be referenced in `ORDER BY` but **not** in `WHERE` (WHERE executes before SELECT).

SQLite string concatenation uses `||`. PostgreSQL supports both `||` and the `CONCAT()` function.


In [ ]:
# Computed expression: full name from two columns
q = """
SELECT
    patient_id,
    first_name || ' ' || last_name   AS full_name,
    province,
    CASE sex
        WHEN 'M' THEN 'Male'
        WHEN 'F' THEN 'Female'
        ELSE          'Other'
    END                              AS sex_label,
    CASE active
        WHEN 1 THEN 'Active'
        ELSE        'Inactive'
    END                              AS status
FROM patients
ORDER BY last_name;
"""
print(pd.read_sql(q, conn).to_string(index=False))


---
## DISTINCT: eliminating duplicate rows

`DISTINCT` applies to the **entire row** — it returns rows where the combination of all selected columns is unique, not just the first column.


In [ ]:
# Distinct provinces in the dataset
print('=== Distinct provinces ===')
print(pd.read_sql(
    'SELECT DISTINCT province FROM patients ORDER BY province', conn
).to_string(index=False))

# DISTINCT on multiple columns — unique (province, sex) combinations
print()
print('=== Distinct (province, sex) combinations ===')
q = 'SELECT DISTINCT province, sex FROM patients ORDER BY province, sex'
print(pd.read_sql(q, conn).to_string(index=False))

# COUNT DISTINCT — how many unique provinces?
print()
print('=== COUNT(DISTINCT province) ===')
q2 = 'SELECT COUNT(DISTINCT province) AS unique_provinces FROM patients'
print(pd.read_sql(q2, conn).to_string(index=False))


---
## WHERE: filtering rows

`WHERE` accepts any expression that evaluates to TRUE or FALSE. Common operators:

| Operator | Example |
|---|---|
| `=` `<>` `<` `>` `<=` `>=` | `WHERE active = 1` |
| `BETWEEN a AND b` | `WHERE dob BETWEEN '1970-01-01' AND '1989-12-31'` (inclusive) |
| `IN (...)` | `WHERE province IN ('NB', 'NS')` |
| `LIKE` | `WHERE last_name LIKE 'Mac%'` (`%` = any chars, `_` = one char) |
| `IS NULL` / `IS NOT NULL` | `WHERE dob IS NOT NULL` |
| `AND`, `OR`, `NOT` | AND binds tighter than OR — use parentheses when mixing |

**NULL rule:** `NULL = NULL` evaluates to NULL (not TRUE). Always use `IS NULL`, never `= NULL`.


In [ ]:
# Simple equality filter
print('=== Active patients only ===')
print(pd.read_sql(
    "SELECT first_name, last_name, province FROM patients WHERE active = 1 ORDER BY last_name",
    conn).to_string(index=False))

# IN operator
print()
print('=== Patients in NB or NS ===')
print(pd.read_sql(
    "SELECT first_name, last_name, province FROM patients WHERE province IN ('NB','NS')",
    conn).to_string(index=False))

# LIKE pattern match
print()
print('=== Last names starting with Mac or Ng ===')
q = """
SELECT first_name, last_name
FROM   patients
WHERE  last_name LIKE 'Mac%' OR last_name LIKE 'Ng%'
"""
print(pd.read_sql(q, conn).to_string(index=False))

# BETWEEN on date strings (works correctly for ISO YYYY-MM-DD)
print()
print('=== Patients born in the 1970s ===')
q2 = """
SELECT first_name, last_name, dob
FROM   patients
WHERE  dob BETWEEN '1970-01-01' AND '1979-12-31'
ORDER BY dob
"""
print(pd.read_sql(q2, conn).to_string(index=False))


---
## LIMIT and ORDER BY

`ORDER BY` determines row sequence. Without it, the database may return rows in any order — this can change between queries as data changes. Always specify `ORDER BY` when order matters.

`LIMIT` caps the result to N rows. Paired together they are the standard pattern for top-N queries.

**PostgreSQL note:** The SQL standard uses `FETCH FIRST n ROWS ONLY` instead of `LIMIT`, but PostgreSQL supports `LIMIT` as well.


In [ ]:
# Order by a single column
print('=== All patients, youngest first ===')
q = 'SELECT first_name, last_name, dob FROM patients ORDER BY dob DESC'
print(pd.read_sql(q, conn).to_string(index=False))

# Multi-column ORDER BY
print()
print('=== Active patients by province then last name ===')
q2 = """
SELECT first_name, last_name, province
FROM   patients
WHERE  active = 1
ORDER BY province ASC, last_name ASC
"""
print(pd.read_sql(q2, conn).to_string(index=False))

# Top-N with LIMIT
print()
print('=== 3 oldest patients ===')
q3 = 'SELECT first_name, last_name, dob FROM patients ORDER BY dob ASC LIMIT 3'
print(pd.read_sql(q3, conn).to_string(index=False))


---
## Putting it together: a realistic query

A complete query combining SELECT expressions, WHERE filtering, ordering, and limiting — the kind you'd write to pull a patient roster for a report.


In [ ]:
# Full working example
q = """
SELECT
    patient_id,
    first_name || ' ' || last_name  AS full_name,
    dob,
    -- Approximate age in years (SQLite — see date_time_functions notebook for full treatment)
    CAST(
        (STRFTIME('%Y','now') - STRFTIME('%Y', dob))
        - CASE
            WHEN STRFTIME('%m-%d','now') < STRFTIME('%m-%d', dob) THEN 1
            ELSE 0
          END
    AS INTEGER)                     AS age,
    province,
    CASE active WHEN 1 THEN 'Active' ELSE 'Inactive' END AS status
FROM   patients
WHERE  active = 1
  AND  province IN ('NB','NS','ON','BC','AB','QC')
ORDER BY province, last_name
LIMIT  8;
"""
print(pd.read_sql(q, conn).to_string(index=False))

conn.close()


---
## Common Pitfalls

**1. Using a SELECT alias in WHERE**
`SELECT cost * 1.13 AS cost_with_tax FROM t WHERE cost_with_tax > 100` fails — WHERE executes before SELECT so the alias doesn't exist yet. Put the expression directly in WHERE, or wrap in a subquery/CTE.

**2. `SELECT *` in production queries**
`SELECT *` pulls every column including large BLOBs or text fields you don't need, wastes I/O, and breaks silently when columns are added or renamed. Name your columns explicitly.

**3. `= NULL` instead of `IS NULL`**
`WHERE dob = NULL` always returns zero rows because `NULL = NULL` evaluates to NULL (unknown), not TRUE. Always write `IS NULL` / `IS NOT NULL`.

**4. DISTINCT on multiple columns vs one column**
`SELECT DISTINCT province, sex` returns unique *(province, sex) pairs*, not unique provinces. If you want unique values of one column, only include that column in the SELECT list.

**5. Assuming result order without ORDER BY**
SQLite and PostgreSQL make no guarantees about row order without an explicit ORDER BY. The same query can return rows in different sequences across executions. Always add ORDER BY when the sequence of results matters.

**6. AND vs OR precedence**
`WHERE province = 'NB' OR province = 'NS' AND active = 1` is parsed as `WHERE province = 'NB' OR (province = 'NS' AND active = 1)` because AND binds tighter than OR. Use parentheses explicitly whenever you mix AND and OR.


---
*sql_methods_library - Samantha McGarrigle*